# Sentiment Classifier Exploration

Quick hands-on exploration of `src/sentiment/classifier.py` before wiring it into `app.py`:

1. Load the lazy-loaded HuggingFace pipeline via `get_classifier()`
2. Run it on a handful of example reviews (positive / negative / neutral / sarcastic)
3. Inspect the raw label + confidence score for each
4. Demonstrate the confidence-threshold "needs review" flagging logic on a small batch


In [ ]:
import sys
from pathlib import Path

# Allow importing `config` and `src` when running this notebook from
# `notebooks/` -- mirrors the sys.path setup used in this repo's other
# project notebooks (e.g. Week 8's 01_eda.ipynb).
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config import get_config
from src.sentiment.classifier import classify, classify_in_batches

cfg = get_config()
print(f"Model:               {cfg.model.model_name}")
print(f"Confidence threshold: {cfg.analysis.confidence_threshold}")


## 1. Example Reviews

Four hand-picked examples, one per interesting case:

- A clearly **positive** review
- A clearly **negative** review
- A genuinely **neutral** review (this is exactly why we need a real 3-class model,
  not a binary one -- see `config.py`'s `ModelConfig` docstring)
- A **sarcastic** review ("Oh great, ANOTHER broken product") -- sentiment models
  struggle with sarcasm because the surface words ("great") point one way while the
  intended meaning points the other. This is Brain Teaser #1 from the Class 2 Lecture.


In [ ]:
example_reviews = [
    "This product exceeded my expectations, the build quality is fantastic!",
    "Terrible experience, it broke after two days and support never replied.",
    "It's fine. Does what it says, nothing more, nothing less.",
    "Oh great, ANOTHER broken product. Just what I needed.",
]

# `classify()` takes the ENTIRE list in one call rather than looping one review
# at a time -- see classifier.py's HIGHLIGHTS on batch processing (this directly
# reuses the HuggingFace pipeline's internal batched tokenize + forward pass).
results = classify(example_reviews)

for r in results:
    print(f"[{r.label:8s}] score={r.score:.3f}  needs_review={r.needs_review!s:5}  {r.text}")


### Reading the results

Watch the **sarcastic** example closely: the model has no notion of tone or irony, it
only sees the words. "Oh great, ANOTHER broken product." contains a positive-sounding
word ("great") right next to negative content ("broken product", "just what I
needed" used ironically). Depending on the model's confidence here, this row is a
good candidate to end up flagged `needs_review=True` -- which is exactly the kind of
case a human reviewer should catch that the model can't reliably resolve on its own.


## 2. Confidence-Threshold Flagging on a Small Batch

`config.py`'s `AnalysisConfig.confidence_threshold` (default `0.6`) decides which
predictions get flagged `needs_review=True`. Below, we classify a slightly larger
batch and split it into "confident" vs. "flagged for review" to see the threshold
in action -- and to sanity-check Brain Teaser #2: what happens if we tighten or
loosen the threshold?


In [ ]:
batch = [
    "Absolutely love it, would buy again in a heartbeat!",
    "The application crashes frequently and lacks essential features.",
    "Average product, priced fairly, nothing that stands out either way.",
    "Customer service was unhelpful and the product stopped working within a week.",
    "Works as described, no complaints so far.",
    "Fast shipping and the packaging was excellent, very happy with this purchase.",
]

batch_results = classify_in_batches(batch)

confident = [r for r in batch_results if not r.needs_review]
flagged = [r for r in batch_results if r.needs_review]

print(f"Confident ({len(confident)}):")
for r in confident:
    print(f"  [{r.label:8s}] score={r.score:.3f}  {r.text}")

print(f"\nFlagged for human review ({len(flagged)}):")
for r in flagged:
    print(f"  [{r.label:8s}] score={r.score:.3f}  {r.text}")


In [ ]:
# Brain Teaser #2: what happens at a much stricter threshold (0.9)
# vs. a much looser one (0.3)? Re-run classification with each and
# compare how many reviews get flagged.
for threshold in (0.3, 0.6, 0.9):
    results_at_threshold = classify(batch, threshold=threshold)
    flagged_count = sum(1 for r in results_at_threshold if r.needs_review)
    print(f"threshold={threshold:.1f} -> {flagged_count}/{len(batch)} flagged for review")


### Takeaways

- A very **high** threshold (0.9) flags almost everything, including predictions the
  model is genuinely confident about -- this defeats the point of automating triage
  and buries human reviewers in false positives.
- A very **low** threshold (0.3) lets genuinely ambiguous predictions (a near-coin-flip
  between two labels) sail through unflagged, so a human never gets a chance to catch
  the model's real uncertainty.
- `0.6` (this project's default, see `config.py`) is a deliberate middle ground --
  above "barely better than random for 3 classes" (~33%) but well below "the model is
  basically certain" (~90%+).
- This same `classify()` / `classify_in_batches()` API is what `app.py`'s Analyze tab
  calls for all three input modes (single review, multi-line paste, CSV upload) --
  nothing in the Streamlit app talks to `transformers` directly.
